In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm


df_final_demo = pd.read_csv('/Users/zahid/Desktop/week 5 project /df_final_demo_clean.csv')
df_final_web_data = pd.read_csv('/Users/zahid/Desktop/week 5 project /df_final_web_data_clean.csv')
df_v1 =pd.read_csv('/Users/zahid/Desktop/week 5 project /df_v2.csv')


###  COMPLETION METRICS BY ALL TIERS

In [2]:


# 1. Group the data by tier and variation group, then sum the visits and completions
tier_summary = df_v1.groupby(['balance_tier', 'Variation']).agg({
    'completed_visits': 'sum',
    'number_of_visits': 'sum'
}).reset_index()

# 2. Calculate the Completion Rate (%)
tier_summary['completion_rate'] = (tier_summary['completed_visits'] / tier_summary['number_of_visits']) * 100

# 3. Pivot the table so Control and Test metrics sit cleanly side-by-side
pivot_summary = tier_summary.pivot(
    index='balance_tier', 
    columns='Variation', 
    values=['number_of_visits', 'completion_rate']
)

# 4. Flatten the multi-level columns for clean reading
pivot_summary.columns = [f"{var}_{metric}" for metric, var in pivot_summary.columns]
pivot_summary = pivot_summary.reset_index()

# 5. Calculate Absolute Increase and Relative Lift for each tier
# (Assumes your 'Variation' values are named 'Control' and 'Test')
if 'Test_completion_rate' in pivot_summary.columns and 'Control_completion_rate' in pivot_summary.columns:
    pivot_summary['Absolute_Increase'] = pivot_summary['Test_completion_rate'] - pivot_summary['Control_completion_rate']
    pivot_summary['Relative_Lift_Pct'] = (pivot_summary['Absolute_Increase'] / pivot_summary['Control_completion_rate']) * 100

# Display the clean table
print("\n=== COMPLETION METRICS BY ALL TIERS ===")

pivot_summary.round(2)


=== COMPLETION METRICS BY ALL TIERS ===


,balance_tier,Control_number_of_visits,Test_number_of_visits,Control_completion_rate,Test_completion_rate,Absolute_Increase,Relative_Lift_Pct
0,Diamond,639.0,753.0,48.04,63.35,15.30,31.85
1,Gold,16324.0,18626.0,48.91,58.05,9.14,18.68
2,Platinum,4294.0,5011.0,47.72,57.95,10.23,21.45
3,Silver,10978.0,12800.0,52.30,59.31,7.02,13.42


### SIGNIFICANCE FOR DIAMOND USERS

In [3]:


# 1. Filter data to include ONLY Diamond users
diamond_df = df_v1[df_v1['balance_tier'].astype(str).str.strip().str.lower() == 'diamond'].copy()

# 2. Group by Variation to count total successful completions and total incomplete visits
contingency_table = diamond_df.groupby('Variation', observed=False).agg(
    completions=('completed_visits', 'sum'),
    visits=('number_of_visits', 'sum')
).reset_index()

# Calculate failed visits (Total Visits minus Successful Completions)
contingency_table['failures'] = contingency_table['visits'] - contingency_table['completions']

# 3. Format data into a 2x2 contingency matrix for SciPy
# Columns: [Completions, Failures] | Rows: [Control, Test]
matrix = contingency_table[['completions', 'failures']].to_numpy()

# 4. Execute the Chi-Square test
chi2, p_value, dof, expected = stats.chi2_contingency(matrix)

print("===  SIGNIFICANCE FOR DIAMOND USERS ===")
for index, row in contingency_table.iterrows():
    rate = (row['completions'] / row['visits']) * 100
    print(f"  • {row['Variation']} Group Completion Rate: {rate:.2f}%")

print(f"\nResults:")
print(f"  • Chi-Square Statistic : {chi2:.4f}")
print(f"  • P-Value              : {p_value:}")

# 5. Output the mathematical verdict (alpha = 0.05)
if p_value < 0.05:
    print("\n📢 Conclusion: Reject H₀ (P < 0.05). The design variation made a statistically significant impact on Diamond users!")
else:
    print("\n❌ Conclusion: Fail to Reject H₀. The performance difference among Diamond users could just be random noise.")


===  SIGNIFICANCE FOR DIAMOND USERS ===
  • Control Group Completion Rate: 48.04%
  • Test Group Completion Rate: 63.35%

Results:
  • Chi-Square Statistic : 32.2855
  • P-Value              : 1.3310226108574474e-08

📢 Conclusion: Reject H₀ (P < 0.05). The design variation made a statistically significant impact on Diamond users!


### User Completion Rate and Visit Completion Rate 

In [10]:


# 1. Load your raw dataset
df_v1 = pd.read_csv('/Users/zahid/Desktop/week 5 project /df_v2.csv')

# 2. Filter out rows with zero visits to prevent division by zero errors
active_sessions = df_v1[df_v1['number_of_visits'] > 0].copy()

# 3. Roll up data to the true client level using 'client_id'
client_rollup = active_sessions.groupby(['Variation', 'client_id'], observed=False).agg(
    client_total_visits=('number_of_visits', 'sum'),
    client_total_completions=('completed_visits', 'sum')
).reset_index()

# 4. Flag unique clients who completed at least 1 visit total
client_rollup['client_completed'] = client_rollup['client_total_completions'] > 0

# 5. Create the primary experiment performance summary
summary_df = client_rollup.groupby('Variation', observed=False).agg(
    total_users=('client_id', 'nunique'),
    completed_users=('client_completed', 'sum'),
    total_visits=('client_total_visits', 'sum'),
    total_completions=('client_total_completions', 'sum')
).reset_index()

# 6. Calculate completion rates as percentages
summary_df['User_Completion_Rate_%'] = (summary_df['completed_users'] / summary_df['total_users']) * 100
summary_df['Visit_Completion_Rate_%'] = (summary_df['total_completions'] / summary_df['total_visits']) * 100

# 7. Extract specific metrics using the exact string keys 'Control' and 'Test'
control_metrics = summary_df[summary_df['Variation'] == 'Control'].iloc[0]
test_metrics = summary_df[summary_df['Variation'] == 'Test'].iloc[0]

# --- Metric A: User Completion Rate Z-Test ---
user_completions = np.array([test_metrics['completed_users'], control_metrics['completed_users']])
user_totals = np.array([test_metrics['total_users'], control_metrics['total_users']])
z_stat_user, p_val_user = sm.stats.proportions_ztest(count=user_completions, nobs=user_totals, alternative='two-sided')

# --- Metric B: Visit Completion Rate Z-Test ---
visit_completions = np.array([test_metrics['total_completions'], control_metrics['total_completions']])
visit_totals = np.array([test_metrics['total_visits'], control_metrics['total_visits']])
z_stat_visit, p_val_visit = sm.stats.proportions_ztest(count=visit_completions, nobs=visit_totals, alternative='two-sided')

# 8. PRINT SUMMARY REPORTS WITH PERCENTAGES
print("=== EXPERIMENT PERFORMANCE REPORT ===")
print(summary_df[[
    'Variation', 'total_users', 'User_Completion_Rate_%', 'total_visits', 'Visit_Completion_Rate_%'
]].to_string(index=False, formatters={'User_Completion_Rate_%': '{:.2f}%'.format, 'Visit_Completion_Rate_%': '{:.2f}%'.format}))

print("\n=== HYPOTHESIS TESTING REPORT ===")
print(f"User Completion Rate (Metric A):")
print(f"  - Control Completion Rate: {control_metrics['User_Completion_Rate_%']:.2f}%")
print(f"  - Test Completion Rate:    {test_metrics['User_Completion_Rate_%']:.2f}%")
print(f"  - Z-Statistic:             {z_stat_user:.4f}")
print(f"  - P-Value:                 {p_val_user:.4e} ({'Significant' if p_val_user < 0.05 else 'Not Significant'} at 95%)")

print(f"\nVisit Completion Rate (Metric B):")
print(f"  - Control Completion Rate: {control_metrics['Visit_Completion_Rate_%']:.2f}%")
print(f"  - Test Completion Rate:    {test_metrics['Visit_Completion_Rate_%']:.2f}%")
print(f"  - Z-Statistic:             {z_stat_visit:.4f}")
print(f"  - P-Value:                 {p_val_visit:.4e} ({'Significant' if p_val_visit < 0.05 else 'Not Significant'} at 95%)")


=== EXPERIMENT PERFORMANCE REPORT ===
Variation  total_users User_Completion_Rate_%  total_visits Visit_Completion_Rate_%
  Control        23526                 65.58%         32235                  49.89%
     Test        26961                 69.29%         37190                  58.58%

=== HYPOTHESIS TESTING REPORT ===
User Completion Rate (Metric A):
  - Control Completion Rate: 65.58%
  - Test Completion Rate:    69.29%
  - Z-Statistic:             8.8927
  - P-Value:                 5.9616e-19 (Significant at 95%)

Visit Completion Rate (Metric B):
  - Control Completion Rate: 49.89%
  - Test Completion Rate:    58.58%
  - Z-Statistic:             22.9355
  - P-Value:                 2.0554e-116 (Significant at 95%)
